# Radius finishing-pass data audit

Companion to `2026-09-09-finishing-pass.md`. Captured findings are in the review and generated coverage scorecard. The rerunnable cells below have not been executed as notebook cells; they inspect local promoted artifacts only, require no credentials and make no external writes.

Public catalog, not raw database places, is the audit population. Source freshness and absence of invalid records do not certify business accuracy.

In [ ]:
from pathlib import Path
import json, sqlite3
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/data/places-client.json').exists())
places = json.loads((root / 'src/data/places-client.json').read_text())
hours = json.loads((root / 'src/data/places-client-hours.json').read_text())
db = sqlite3.connect(':memory:')
db.execute('create table public_places (slug text, municipality text, category text, description text, geometry_present integer, hours_verified integer)')
db.executemany('insert into public_places values (?, ?, ?, ?, ?, ?)', [(p.get('slug'), p.get('municipality'), p.get('category'), p.get('short_blurb'), bool(p.get('geom')), bool(p.get('hours_verified'))) for p in places])
print('Public hours artifact rows:', len(hours))


In [ ]:
queries = {
    'identity_and_structure': '''select count(*) as places, count(distinct slug) as unique_slugs, count(distinct municipality) as municipalities, sum(case when coalesce(slug, '') = '' then 1 else 0 end) as blank_slugs, sum(case when geometry_present = 0 then 1 else 0 end) as missing_geometry from public_places''',
    'municipality_coverage': '''select municipality, count(*) as places from public_places group by municipality order by places desc''',
    'brunswick_coffee': '''select slug, category, description from public_places where municipality = 'Brunswick' and category = 'coffee' order by slug'''
}
for label, query in queries.items():
    result = db.execute(query)
    print(label, [c[0] for c in result.description])
    print(result.fetchall())


## Additional reproducible checks

Run the repository's `npm run gates`, `npm run coverage:scorecard`, and `npm run data:release:check` for the actual county polygon, copy provenance, publishable-photo and freshness policies. The simple geometry-presence query above is not a polygon validation.

Live event and transit findings in the review are point-in-time observations from September 9, 2026. They are not outputs of this notebook and must be rechecked before presenting them as current. No direct production data cleanup is authorized by these cells.
